In [1]:
import mlflow
from datasets import load_dataset
from src.embedder.sparse import SparseEmbedder
from src.datasource.sparse import SparseDatasource
from src.utils import evaluate_model, load_test_data

/Users/mikhailkoutun/PycharmProjects/searchEngine/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
queries_dataset = load_dataset("CoIR-Retrieval/cosqa", "queries")["queries"]
corpus_dataset = load_dataset("CoIR-Retrieval/cosqa", "corpus")["corpus"]
default_dataset = load_dataset("CoIR-Retrieval/cosqa", "default")
test_corpus = [function for partition, function in zip(corpus_dataset["partition"], corpus_dataset["text"]) if
               partition == "test"]
test_queries = [query for partition, query in zip(queries_dataset["partition"], queries_dataset["text"]) if
                partition == "test"]


In [5]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("CoIR-Retrieval/sparse")

sparse_models = [SparseEmbedder("Qdrant/bm25")]
for sparse_model in sparse_models:
    run_name = sparse_model.model_name
    with mlflow.start_run(run_name=run_name) as run:

        mlflow.log_params({
            "sparse_model" : sparse_model.model_name,
        })

        db = SparseDatasource(sparse_model)
        load_test_data(db, "code-test-sparse", test_corpus, True)
        recall, mrr, ndcg = evaluate_model(db, "code-test-sparse", test_queries, test_corpus)

        mlflow.log_metrics({
            "recall:10" : float(recall),
            "mrr:10" : float(mrr),
            "ndcg:10" : float(ndcg),
        })

2025/12/27 19:36:22 INFO mlflow.tracking.fluent: Experiment with name 'CoIR-Retrieval/sparse' does not exist. Creating a new experiment.


🏃 View run Qdrant/bm25 at: http://127.0.0.1:5000/#/experiments/3/runs/43dfcee686144c04bf56ab87a1dd27cf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
